# Supervisor review: pediatric cancer drug-repurposing graph model

**Repository:** [Maxmahesh-byte/drug-repurposing](https://github.com/Maxmahesh-byte/drug-repurposing)  
**Review branch:** `feature/multimodal-drug-cell-gnn`  
**Notebook purpose:** a single, reproducible entry point for reviewing the research question, data provenance, graph design, model implementation, evaluation, available results, and remaining work.

> **Important status note (4 November 2026):** the ChEMBL/RDKit chemical-feature pipeline and its optional GNN ablation have been committed to this branch. The code has not yet been run end-to-end on the project's real data in this review. This notebook therefore distinguishes completed historical benchmark results from new code whose runtime and predictive effect still need validation.

This is a review notebook, not a model-training notebook. Its default behaviour is read-only: it inspects artifacts and reports their availability rather than launching long training jobs.

## 1. Executive summary

The project investigates whether curated drug-target evidence and cell-line molecular context can help predict **measured in-vitro GDSC2 drug response**.

The supervised endpoints are:

- **LN(IC50):** natural logarithm of the fitted half-maximal inhibitory concentration.
- **AUC:** area under the fitted dose-response curve.

The existing benchmark is a **known-drug, held-out-cell-line, transductive** evaluation. It asks whether a model trained on some cell lines can predict measured GDSC2 response for other cell lines, while the evaluated drug set is already represented in the training graph.

It does **not** establish patient benefit, clinical efficacy, treatment suitability, or generalisation to unseen drugs.

### What a supervisor can check here

1. Data scale, matching coverage, identity rules, and source boundaries.
2. The response-free heterogeneous graph and its evidence relations.
3. The model architecture and the way chemical information is added.
4. Frozen split design, leakage safeguards, baselines, and historical five-seed ablation results.
5. Whether newly expected data artifacts exist and whether they agree with their manifests.
6. Remaining implementation and validation tasks.

## 2. Research design and interpretation

### Current prediction task

| Element | Definition |
|---|---|
| Drug domain | Known GDSC2 compounds represented in the graph |
| Cell-line domain | GDSC2 cell lines with audited exact-name expression matches |
| Inputs | Drug identity/evidence, cell-line expression, and optionally chemical structure features |
| Targets | Measured GDSC2 LN(IC50) and AUC |
| Outer evaluation | Five frozen folds grouped by Sanger model ID |
| Generalisation tested | Held-out cell lines for known drugs |
| Not tested by this benchmark | Unseen-drug transfer, patient outcomes, clinical response |

### Data domains must remain distinct

- **GDSC2** supplies in-vitro single-agent dose-response measurements for cancer cell lines.
- **CCLE / E-MTAB-2770** supplies expression measurements used for exact-name-matched cell-line context.
- **ChEMBL** supplies compound identity, structure, and curated mechanism/target annotations.
- **OpenPedCan** supplies pediatric tumour cohort context. Its patient biospecimens are not GDSC cell lines and must not be described as matched cell-line measurements.

Shared gene identifiers can connect evidence in a graph. They do not make a patient biospecimen and a cell line the same biological sample.

## 3. Repository map: where to inspect the implementation

The following links point to the review branch so the reviewer can open the implementation directly without browsing the whole repository.

| Component | Main file |
|---|---|
| Compact heterogeneous drug/context graph | [src/training/gdsc2_pair_gnn.py](https://github.com/Maxmahesh-byte/drug-repurposing/blob/feature/multimodal-drug-cell-gnn/src/training/gdsc2_pair_gnn.py) |
| GDSC2 response-pair preparation | [scripts/ai/prepare_gdsc2_gnn_dataset.py](https://github.com/Maxmahesh-byte/drug-repurposing/blob/feature/multimodal-drug-cell-gnn/scripts/ai/prepare_gdsc2_gnn_dataset.py) |
| Frozen-fold GNN evaluation and ablations | [scripts/ai/run_gdsc2_pair_gnn_ablations.py](https://github.com/Maxmahesh-byte/drug-repurposing/blob/feature/multimodal-drug-cell-gnn/scripts/ai/run_gdsc2_pair_gnn_ablations.py) |
| ChEMBL molecule/target ingestion | [src/ingestion/chembl_targets.py](https://github.com/Maxmahesh-byte/drug-repurposing/blob/feature/multimodal-drug-cell-gnn/src/ingestion/chembl_targets.py) |
| **New:** ChEMBL structures → RDKit chemical features | [scripts/ai/build_gdsc2_chemical_features.py](https://github.com/Maxmahesh-byte/drug-repurposing/blob/feature/multimodal-drug-cell-gnn/scripts/ai/build_gdsc2_chemical_features.py) |
| Coordinated multimodal upgrade plan | [docs/MULTIMODAL_UPGRADE_PLAN.md](https://github.com/Maxmahesh-byte/drug-repurposing/blob/feature/multimodal-drug-cell-gnn/docs/MULTIMODAL_UPGRADE_PLAN.md) |
| Tracking issue | [Issue #13](https://github.com/Maxmahesh-byte/drug-repurposing/drug-repurposing/issues/13) |

The model implementation uses PyTorch Geometric's **RGCNConv** (relational graph convolution). The current branch's model class defines five message-passing layers. The new chemical ablation adds a chemical feature encoder to drug nodes; it does not replace the existing heterogeneous graph with a new graph from scratch.

## 4. Data and coverage audit

The table below records the project's previously reported audit snapshot. The notebook's next cells check whether the corresponding local manifests and data artifacts are available so these summary values can be refreshed from the checked-out files.

| Audit item | Reported value | Interpretation |
|---|---:|---|
| GDSC2 response rows after requiring both endpoints | 242,036 | Measured drug–cell-line response records |
| GDSC2 cell lines | 969 | Before exact-name expression matching |
| GDSC2 drugs | 295 | From the project workbook filter; distinct from other live dataset summaries |
| Exact-name expression-matched cell lines | 528 / 969 | Matched by the audited crosswalk |
| Matched response rows | 133,929 | Rows retained for expression-based modelling |
| Drugs with exact ChEMBL mapping | 167 / 295 | Exact preferred-name or synonym mapping |
| Mapped drugs with direct human mechanism evidence | 97 | Curated direct target evidence |
| Mapped drugs without direct human mechanism evidence | 70 | Mapping exists; target evidence is absent |
| Unmatched drugs | 126 | No unambiguous exact mapping |
| Ambiguous drugs | 2 | Excluded rather than guessed |
| Distinct ChEMBL molecules for mapped pairs | 159 | Molecule IDs represented by the mappings |
| Graph V2 nodes / edges | 271,466 / 7,974,712 | Existing typed PyG graph artifact |
| Legacy graph type/schema mismatch edges retained during migration | 168 | Preserved under legacy mismatch relation names; requires interpretation |

**Why mapping states matter:** a drug name mapped to a ChEMBL molecule is an identity-resolution observation, not proof that the compound has a direct human mechanism record. An absent mechanism is not evidence of no biological activity.

## 5. Graph evidence and leakage controls

### Drug-side evidence path

`GDSC drug → ChEMBL molecule → direct mechanism target → gene`

The compact drug graph contains the following core forward relations:

1. `drug — exact_name_chembl_mapping → chembl_molecule`
2. `chembl_molecule — chembl_direct_mechanism → target`
3. `target — has_chembl_gene_symbol → gene`

The current pediatric-context variant may also include OpenPedCan sample/tumour/fusion context relations that touch genes in the drug-target subgraph. This is a disease-context signal, **not** a patient-to-cell-line identity match.

### Leakage safeguards to verify

- GDSC2 response labels are stored in a separate response-pair table, not copied into graph message-passing edges.
- LN(IC50) and AUC must not be graph node/edge input features.
- The outer-test cell lines must not contribute to fold-fitted expression selection or scaling.
- Endpoint scaling is fitted on training response rows only.
- Drug identity/structure features are label-free; their alignment must be checked against graph drug-node order.
- Patient biospecimens must not be relabelled as GDSC cell lines.

**Scientific limitation:** evidence presence is not the same as target activity in a particular cell line, and a graph connection is not by itself a predictive association.

In [ ]:
from pathlib import Path
import json
import hashlib
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Find the repository root without assuming the notebook's launch directory.
def find_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in (candidate, *candidate.parents):
        if (path / "scripts" / "ai").is_dir() and (path / "src").is_dir():
            return path
    raise FileNotFoundError(
        "Could not find repository root. Open this notebook from inside the "
        "drug-repurposing checkout, or set PROJECT_ROOT manually."
    )

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", find_project_root())).resolve()
DATA = PROJECT_ROOT / "data"
REVIEW_OUTPUT = DATA / "ai" / "supervisor_review"
REVIEW_OUTPUT.mkdir(parents=True, exist_ok=True)

def artifact_status(path):
    path = Path(path)
    return {
        "path": str(path.relative_to(PROJECT_ROOT)) if path.is_relative_to(PROJECT_ROOT) else str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
    }

def load_json_if_present(path):
    path = Path(path)
    if not path.is_file():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Notebook runs in read-only inspection mode; it does not start model training.")

## 6. Artifact availability check

This cell checks the expected data products and model outputs. A missing file is reported as **not present**, not treated as a failed experiment. Some artifacts are generated only after running the preparation or modelling scripts.

In [ ]:
expected_artifacts = [
    PROJECT_ROOT / "data/processed/graph_v2/typed_graph_v2.pt",
    PROJECT_ROOT / "data/ai/response_dataset/gdsc2_matched_response_pairs.csv",
    PROJECT_ROOT / "data/ai/response_dataset/gdsc2_matched_expression_features.npz",
    PROJECT_ROOT / "data/ai/response_dataset/gdsc2_matched_response_dataset_summary.json",
    PROJECT_ROOT / "data/raw/chembl/gdsc2_drug_molecule_mappings.csv",
    PROJECT_ROOT / "data/ai/chemical_features/gdsc2_chemical_features.npz",
    PROJECT_ROOT / "data/ai/chemical_features/gdsc2_chemical_feature_audit.csv",
    PROJECT_ROOT / "data/ai/chemical_features/gdsc2_chemical_feature_manifest.json",
]
artifact_table = pd.DataFrame([artifact_status(p) for p in expected_artifacts])
display(artifact_table)

# Discover evaluation outputs without assuming a single output directory.
metrics_candidates = sorted(DATA.glob("ai/**/gdsc2_pair_gnn_ablation_metrics.json"))
prediction_candidates = sorted(DATA.glob("ai/**/gdsc2_pair_gnn_ablation_oof_predictions.csv"))
print(f"Found {len(metrics_candidates)} ablation metrics file(s).")
for p in metrics_candidates:
    print(" -", p.relative_to(PROJECT_ROOT))
print(f"Found {len(prediction_candidates)} OOF prediction file(s).")
for p in prediction_candidates:
    print(" -", p.relative_to(PROJECT_ROOT))

## 7. Refreshable response-dataset summary

When the preparation summary exists, this cell reads its row counts, matched model counts, fold information, input hashes, and validation statements. It does not assume that a prior local run has been repeated on the current checkout.

In [ ]:
response_summary_path = DATA / "ai/response_dataset/gdsc2_matched_response_dataset_summary.json"
response_summary = load_json_if_present(response_summary_path)

if response_summary is None:
    print("Response dataset summary not found:", response_summary_path.relative_to(PROJECT_ROOT))
else:
    overview = {
        "status": response_summary.get("status"),
        "response_rows": response_summary.get("response_rows"),
        "matched_models": response_summary.get("matched_models"),
        "drugs": response_summary.get("drugs"),
        "genes": response_summary.get("genes"),
        "five_folds_present": response_summary.get("validation", {}).get("five_folds_present"),
        "exact_name_crosswalk_only": response_summary.get("validation", {}).get("exact_name_crosswalk_only"),
        "response_labels_in_expression_features": response_summary.get("validation", {}).get("response_labels_in_expression_features"),
    }
    display(pd.Series(overview, name="Prepared response dataset"))
    fold_info = response_summary.get("folds", {})
    if fold_info:
        display(pd.DataFrame.from_dict(fold_info, orient="index").rename_axis("fold"))
    display(Markdown("**Important limitations recorded by the preparation script:**"))
    for item in response_summary.get("important_limitations", []):
        print("•", item)

## 8. Chemical feature pipeline

The newly committed script `scripts/ai/build_gdsc2_chemical_features.py`:

1. Reads the audited GDSC-to-ChEMBL mapping table.
2. Retrieves each mapped ChEMBL molecule record and canonical SMILES.
3. Calculates a 2,048-bit Morgan fingerprint (radius 2) and eight molecular descriptors using RDKit.
4. Records canonical SMILES, InChI, InChIKey, mapping provenance, and structure availability in an audit CSV.
5. Writes a compressed NPZ whose `drug_ids[i]` and `features[i]` are aligned to the exact order of the graph's drug nodes.

Unmapped drugs are retained in graph order with zero-valued chemical vectors and an explicit `structure_available=0` indicator. This avoids silently dropping drugs or inventing molecular identities.

### Output files

- `gdsc2_chemical_features.npz`
- `gdsc2_chemical_feature_audit.csv`
- `gdsc2_chemical_feature_manifest.json`

The feature pipeline is label-free. It does not use IC50/AUC values.

In [ ]:
chemical_npz = DATA / "ai/chemical_features/gdsc2_chemical_features.npz"
chemical_audit_path = DATA / "ai/chemical_features/gdsc2_chemical_feature_audit.csv"
chemical_manifest_path = DATA / "ai/chemical_features/gdsc2_chemical_feature_manifest.json"

chemical_manifest = load_json_if_present(chemical_manifest_path)
if chemical_manifest:
    display(pd.Series({
        "graph_drug_count": chemical_manifest.get("graph_drug_count"),
        "mapped_graph_drug_count": chemical_manifest.get("mapped_graph_drug_count"),
        "unmapped_graph_drug_count": chemical_manifest.get("unmapped_graph_drug_count"),
        "structure_available_count": chemical_manifest.get("structure_available_count"),
        "structure_missing_count": chemical_manifest.get("structure_missing_count"),
        "feature_width": chemical_manifest.get("feature_width"),
        "fingerprint": chemical_manifest.get("fingerprint"),
        "response_labels_used": chemical_manifest.get("response_labels_used"),
    }, name="Chemical feature manifest"))

if chemical_npz.is_file():
    with np.load(chemical_npz, allow_pickle=False) as archive:
        chemical_ids = archive["drug_ids"].astype(str)
        chemical_matrix = archive["features"].astype(np.float32)
        chemical_names = archive["feature_names"].astype(str)
    print("Feature matrix shape:", chemical_matrix.shape)
    print("Drug ID count:", len(chemical_ids))
    print("Feature-name count:", len(chemical_names))
    print("All finite:", bool(np.isfinite(chemical_matrix).all()))
    print("Unique drug IDs:", bool(len(set(chemical_ids.tolist())) == len(chemical_ids)))
    if chemical_matrix.ndim == 2 and chemical_matrix.shape[1] == len(chemical_names):
        display(pd.DataFrame({
            "feature": chemical_names[-9:],
            "example_mean": chemical_matrix[:, -9:].mean(axis=0),
            "example_sd": chemical_matrix[:, -9:].std(axis=0),
        }))
else:
    print("Chemical feature NPZ not yet generated. Run the builder script before the optional chemical ablation.")

## 9. Model architecture and current implementation status

### Current branch model

- **Graph encoder:** five-layer R-GCN using relation-specific message passing.
- **Node representation:** learned entity and node-type embeddings.
- **Cell-line encoder:** multilayer perceptron over expression features.
- **Chemical encoder (optional):** linear encoder over Morgan bits and descriptors, added to the drug-node representation.
- **Pair head:** receives drug embedding, cell embedding, elementwise product, and absolute difference; predicts two endpoints (LN(IC50), AUC).

The chemical model is enabled as a distinct `chemical_evidence` ablation. It uses the gene-evidence graph relations plus structure features, so its comparison tests the incremental contribution of chemical structure features over the corresponding graph model.

### Seven planned workstreams: status at this branch

| Workstream | Current state | What remains |
|---|---|---|
| 1. Evidence-preserving graph foundation | Existing typed PyG graph and compact evidence extraction are used | Rebuild lost/aggregated observations from raw sources with namespaced IDs and provenance |
| 2. Drug identity and chemistry | ChEMBL mappings exist; RDKit feature builder and optional model input are committed | Run, audit structure coverage, test model input end-to-end |
| 3. Cell-line-matched omics | Exact-name-matched expression is used | Audit and integrate matched mutation/CNV/fusion data with stable cell-model IDs |
| 4. Pediatric context and pathways | OpenPedCan context relation family exists in the graph path | Validate sample-level provenance and add curated pathway context |
| 5. Drug–cell interaction | Concatenation/product/absolute-difference pair head exists | Test target/pathway-conditioned gating or attention |
| 6. Architecture comparison | R-GCN baseline exists | Compare attention/HGT alternatives on identical inputs and splits |
| 7. Evaluation | Five-fold held-out-cell-line benchmark and baselines exist | Add drug-held-out and cold drug+cell evaluation; uncertainty/calibration analyses |

The status table is an implementation snapshot, not a claim that every proposed workstream is complete.

## 10. Evaluation design and metrics

### Split protocol

The existing controlled benchmark uses five frozen folds grouped by **Sanger model ID**. For each outer fold:

1. The outer test fold is held aside.
2. An inner group-based validation split selects the epoch.
3. Expression feature selection and scaling are fitted using permitted training cell lines only.
4. The final model is reinitialised and refitted on every response row from the full outer-training fold.
5. Predictions are generated for the untouched outer-test fold.

The same outer and inner splits are shared across the compared variants. Predictions are returned to original target units before reporting errors.

### Metrics

- **MAE** and **RMSE**: prediction error in original endpoint units (lower is better).
- **R²**: variance-explained summary; can be negative on held-out data.
- For model comparisons, paired differences are computed on the same held-out records.

The current primary benchmark is not a drug-held-out benchmark. It is transductive with respect to the known drug set.

## 11. Previously completed five-seed ablation results

The following values are from the **corrected full-mapping, five-seed evidence-layer ablation** previously run on the matched response set (133,929 rows per seed, frozen five-fold GroupKFold by Sanger model ID). They are included as historical results for supervisor review; they are **not** results from the newly added chemical-feature model.

For each row, the reported difference is **Model A − Model B** in the endpoint error metric. Negative values favour Model A; positive values favour Model B. SD describes variation across the five seed-level differences. These are descriptive paired comparisons, **not** p-values or proof of statistical significance.

In [ ]:
historical_results = pd.DataFrame([
    # Endpoint, A - B, mean, SD, seed consistency note
    ("LN(IC50)", "ChEMBL mapping − drug-ID-only",  0.003052, 0.014539, "Mixed; one seed has a large positive difference"),
    ("LN(IC50)", "Target evidence − ChEMBL mapping", -0.002256, 0.009526, "Mixed across seeds"),
    ("LN(IC50)", "Gene evidence − target evidence",  0.001627, 0.005764, "Mixed across seeds"),
    ("LN(IC50)", "Gene evidence − drug-ID-only",    0.002423, 0.018984, "Mixed; graph better in 2/5 seeds"),
    ("LN(IC50)", "Gene evidence − expression-only", -0.987099, np.nan, "Graph substantially lower error"),
    ("LN(IC50)", "Full graph − drug+cancer median", 0.017535, 0.006920, "Graph worse in 5/5 seeds"),
    ("LN(IC50)", "Full graph − expression Ridge",   0.039843, 0.006920, "Graph worse in 5/5 seeds"),
    ("AUC", "ChEMBL mapping − drug-ID-only",        -0.000517, 0.000667, "Graph lower error in 4/5 seeds"),
    ("AUC", "Target evidence − ChEMBL mapping",     -0.000134, 0.000568, "Graph lower error in 3/5 seeds"),
    ("AUC", "Gene evidence − target evidence",       0.000040, 0.000246, "Approximately 3/5 seeds favour gene evidence"),
    ("AUC", "Gene evidence − drug-ID-only",         -0.000611, 0.000911, "Graph lower error in 4/5 seeds"),
    ("AUC", "Gene evidence − expression-only",      -0.039663, np.nan, "Graph substantially lower error"),
    ("AUC", "Full graph − drug+cancer median",       0.003052, np.nan, "Graph worse in 5/5 seeds"),
    ("AUC", "Full graph − expression Ridge",        0.001108, np.nan, "Graph worse in 5/5 seeds"),
], columns=["endpoint", "comparison (A − B)", "mean_error_difference", "SD_across_seeds", "reported_consistency"])
display(historical_results)

display(Markdown(
    "**Interpretation:** ChEMBL evidence can provide useful signal in some comparisons, "
    "especially for AUC, but the benefit is not uniform across seeds or endpoints. "
    "The full graph did not outperform the drug+cancer median or expression Ridge baseline "
    "in these reported comparisons. Adding more graph evidence is therefore not assumed "
    "to improve prediction. The chemical-feature ablation is a new hypothesis to test, "
    "not a result that can be claimed in advance."
))

## 12. Load the most recent available experiment outputs

This cell scans for saved metrics JSON files created by the ablation runner. If several runs exist, it lists them and reads the most recently modified file for a compact review. It does not overwrite any results.

In [ ]:
if metrics_candidates:
    metrics_candidates = sorted(metrics_candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    selected_metrics_path = metrics_candidates[0]
    latest_metrics = load_json_if_present(selected_metrics_path)
    print("Selected metrics:", selected_metrics_path.relative_to(PROJECT_ROOT))
    print("Run status:", latest_metrics.get("status"))
    print("Experiment:", latest_metrics.get("experiment"))
    print("Variants declared:", list(latest_metrics.get("variants", {}).keys()))
    print("Chemical structure features used:",
          latest_metrics.get("graph_ablation_design", {}).get("chemical_structure_features_used"))
    overall_metrics = latest_metrics.get("overall_metrics", {})
    for endpoint, endpoint_metrics in overall_metrics.items():
        print("\nEndpoint:", endpoint)
        display(pd.DataFrame(endpoint_metrics).T)
else:
    latest_metrics = None
    selected_metrics_path = None
    print("No saved GNN ablation metrics JSON was found under data/ai/.")
    print("This means the notebook cannot report a run from the checked-out data directory; it does not imply that the code failed.")

## 13. Optional OOF prediction inspection

When an out-of-fold prediction CSV exists, the reviewer can inspect its coverage and recalculate basic metrics from the saved predictions. The notebook selects the newest matching file and checks that curve IDs are unique. It does not fit a model.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if prediction_candidates:
    prediction_candidates = sorted(prediction_candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    selected_prediction_path = prediction_candidates[0]
    oof = pd.read_csv(selected_prediction_path)
    print("Selected OOF file:", selected_prediction_path.relative_to(PROJECT_ROOT))
    print("Rows:", len(oof))
    print("Unique curves:", oof["curve_id"].nunique() if "curve_id" in oof else "curve_id column missing")
    if "curve_id" in oof and oof["curve_id"].duplicated().any():
        raise ValueError("OOF prediction file has duplicate curve IDs.")
    prediction_columns = [c for c in oof.columns if c.startswith("pred_")]
    print("Prediction columns:", prediction_columns)

    calculated = []
    for target in ("ln_ic50", "auc"):
        actual_col = f"actual_{target}"
        if actual_col not in oof:
            continue
        actual = pd.to_numeric(oof[actual_col], errors="coerce").to_numpy(dtype=float)
        for pred_col in prediction_columns:
            if not pred_col.endswith(f"_{target}"):
                continue
            pred = pd.to_numeric(oof[pred_col], errors="coerce").to_numpy(dtype=float)
            mask = np.isfinite(actual) & np.isfinite(pred)
            if mask.any():
                calculated.append({
                    "endpoint": target,
                    "model": pred_col.removeprefix("pred_").removesuffix(f"_{target}"),
                    "n": int(mask.sum()),
                    "MAE": mean_absolute_error(actual[mask], pred[mask]),
                    "RMSE": mean_squared_error(actual[mask], pred[mask]) ** 0.5,
                    "R2": r2_score(actual[mask], pred[mask]) if mask.sum() > 1 else np.nan,
                })
    if calculated:
        calculated_metrics = pd.DataFrame(calculated).sort_values(["endpoint", "MAE"])
        display(calculated_metrics)
else:
    oof = None
    selected_prediction_path = None
    print("No saved OOF prediction CSV was found. Run the evaluation script before using this section.")

## 14. Reproducibility and integrity checklist

Use this checklist when reviewing a run. A checked item should be supported by a saved artifact, manifest, source code, or execution record—not merely by a sentence in a report.

- [ ] The response-pair CSV and expression NPZ have matching Sanger model IDs and documented hashes.
- [ ] Each cell line belongs to exactly one frozen outer fold.
- [ ] No response edge or response-derived feature enters graph message passing.
- [ ] Fold-specific expression selection/scaling uses training cell lines only.
- [ ] Target scaling uses training response rows only.
- [ ] Drug-node order in the chemical NPZ exactly matches graph drug-node order.
- [ ] Unmapped drugs and missing structures are explicitly flagged, not silently imputed as known structures.
- [ ] All compared models use the same outer/inner split assignments.
- [ ] OOF predictions cover the expected held-out curves exactly once.
- [ ] Baselines are joined by curve ID and checked for fold, cell-line ID, and measured-target agreement.
- [ ] The chemical-feature model has been run on the same folds/seeds before making any claim about its performance.
- [ ] A drug-held-out evaluation is reported separately before claiming chemical transfer to unseen compounds.

### Important remaining limitations

1. Current results address known drugs and held-out cell lines, not unseen-drug prediction.
2. OpenPedCan tumour context is not a substitute for matched molecular profiles of the GDSC/CCLE cell lines.
3. ChEMBL target annotations are curated evidence, not proof of target engagement in the tested cell line.
4. The newly committed chemical-feature code still requires an end-to-end run and independent artifact checks.
5. No clinical response or patient-benefit claim is supported by these in-vitro data.

## 15. Suggested supervisor review questions

1. Is the distinction between patient biospecimens and cell-line response data maintained clearly throughout the graph and interpretation?
2. Are exact-name ChEMBL mappings and unresolved/ambiguous compound identities handled conservatively enough?
3. Does the evidence graph encode biologically meaningful drug–target–gene context without response-label leakage?
4. Are the split and preprocessing safeguards sufficient for the stated held-out-cell-line benchmark?
5. Which additional cell-line-matched omics and pathway resources should be prioritised?
6. Should the next model experiment focus first on chemical fingerprints, a target-conditioned interaction head, or a drug-held-out test?
7. What evidence would be required before describing any finding as relevant to pediatric patient treatment?

### Recommended next milestone

First run the new chemical-feature builder, inspect its audit CSV and manifest, and then run the optional `chemical_evidence` ablation on the exact same folds and seeds as the baseline. Compare the paired held-out predictions to the existing graph model, expression Ridge, and drug+cancer median. Do not promote the new model based on a single seed or a single endpoint.

## 16. Run instructions

Open this notebook from the repository root in JupyterLab, VS Code, or Google Colab after cloning the branch. The notebook itself only inspects outputs; the following commands generate the artifacts it can review.

### A. Prepare the response-pair dataset (only if not already prepared)

```bash
python scripts/ai/prepare_gdsc2_gnn_dataset.py
```

### B. Build chemical features

```bash
python scripts/ai/build_gdsc2_chemical_features.py \\
  --mapping data/raw/chembl/gdsc2_drug_molecule_mappings.csv \\
  --graph data/processed/graph_v2/typed_graph_v2.pt \\
  --output-dir data/ai/chemical_features
```

### C. Run the controlled ablation

```bash
python scripts/ai/run_gdsc2_pair_gnn_ablations.py \\
  --graph-file data/processed/graph_v2/typed_graph_v2.pt \\
  --chemical-features data/ai/chemical_features/gdsc2_chemical_features.npz \\
  --output-dir data/ai/gdsc2_chemical_ablation \\
  --seed 42
```

The notebook's output-discovery cell scans `data/ai/` for the metrics JSON and OOF prediction CSV. For a multi-seed study, run the same experiment with the pre-specified seed set and preserve separate outputs per seed.

---

**Review principle:** report what was measured, what was implemented, what was validated, and what remains a hypothesis as separate things. A better model score is not guaranteed by adding more modalities or graph complexity.